In [1]:
# Build and verify the locked final feature pipeline for all downstream modeling.

import pandas as pd
import numpy as np

from catboost import CatBoostRegressor

# Load raw data
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

# Normalize product category exactly as established during EDA
train["product_category_clean"] = (
    train["product_category"].str.strip().str.lower()
)
test["product_category_clean"] = (
    test["product_category"].str.strip().str.lower()
)

# Separate target and remove identifiers/raw duplicate category
X_final = train.drop(
    columns=["total_sales", "id", "product_category"]
).copy()

X_test_final = test.drop(
    columns=["id", "product_category"]
).copy()

y = train["total_sales"].copy()

# Locked categorical feature definition
cat_features = [
    "product_code",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format",
]

# Make categorical missingness explicit for CatBoost
for col in cat_features:
    X_final[col] = X_final[col].fillna("Missing").astype(str)
    X_test_final[col] = X_test_final[col].fillna("Missing").astype(str)

# Verify train/test feature alignment
assert list(X_final.columns) == list(X_test_final.columns)
assert len(cat_features) == 7
assert "id" not in X_final.columns
assert "product_category" not in X_final.columns
assert "total_sales" not in X_final.columns

print("Final train shape:", X_final.shape)
print("Final test shape:", X_test_final.shape)
print("Target shape:", y.shape)
print("Categorical features:", cat_features)
print("\nFeature dtypes:")
print(X_final.dtypes)

Final train shape: (6818, 11)
Final test shape: (1705, 11)
Target shape: (6818,)
Categorical features: ['product_code', 'fat_content', 'product_category_clean', 'store_code', 'store_size', 'store_location_tier', 'store_format']

Feature dtypes:
product_code               object
product_weight_kg         float64
fat_content                object
shelf_visibility          float64
product_price             float64
store_code                 object
store_age_years             int64
store_size                 object
store_location_tier        object
store_format               object
product_category_clean     object
dtype: object


In [2]:
# Build the finalized train/test datasets directly from the raw files and save them as reproducible CSV artifacts.

import pandas as pd

train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

# Normalize the product category using the locked preprocessing rule.
train["product_category_clean"] = train["product_category"].str.strip().str.lower()
test["product_category_clean"] = test["product_category"].str.strip().str.lower()

final_features = [
    "product_code",
    "product_weight_kg",
    "fat_content",
    "shelf_visibility",
    "product_price",
    "store_code",
    "store_age_years",
    "store_size",
    "store_location_tier",
    "store_format",
    "product_category_clean",
]

train_final = train[final_features + ["total_sales"]].copy()
test_final = test[final_features].copy()

# Preserve the exact categorical representation used by the frozen CatBoost model.
cat_features = [
    "product_code",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format",
]

for col in cat_features:
    train_final[col] = train_final[col].fillna("__MISSING__").astype(str)
    test_final[col] = test_final[col].fillna("__MISSING__").astype(str)

train_final.to_csv("../data/train_final.csv", index=False)
test_final.to_csv("../data/test_final.csv", index=False)

print("train_final.csv:", train_final.shape)
print("test_final.csv:", test_final.shape)
print("Train columns:", train_final.columns.tolist())
print("Test columns:", test_final.columns.tolist())

train_final.csv: (6818, 12)
test_final.csv: (1705, 11)
Train columns: ['product_code', 'product_weight_kg', 'fat_content', 'shelf_visibility', 'product_price', 'store_code', 'store_age_years', 'store_size', 'store_location_tier', 'store_format', 'product_category_clean', 'total_sales']
Test columns: ['product_code', 'product_weight_kg', 'fat_content', 'shelf_visibility', 'product_price', 'store_code', 'store_age_years', 'store_size', 'store_location_tier', 'store_format', 'product_category_clean']
